<a href="https://colab.research.google.com/github/tevfikaytekin/CMP2003/blob/main/content_based.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content-based Recommendation

Content-based recommender algorithms use the content of the items for making a recommendation. For example, for movie recommendation movie contents such as plot summary, director, casting, jenres, release date, etc. And user content such as previously watched movies, gender, age, etc. can be used to find out which movies can be recommended to the users.


In [75]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from scipy.sparse import csr_matrix
from collections import Counter
from sklearn.metrics import pairwise_distances
from operator import itemgetter
import copy
import heapq
import sys, os
import pickle
import itertools
import operator
from tqdm.notebook import tqdm

In [76]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip

--2026-10-06 06:59:28--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip.2’

ml-latest-small.zip 100%[===================>] 955.28K   868KB/s    in 1.1s    

2026-10-06 06:59:30 (868 KB/s) - ‘ml-latest-small.zip.2’ saved [978202/978202]

Archive:  ml-latest-small.zip
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


### Movielens ml-latest-small dataset

In [77]:
with open('/content/ml-latest-small/README.txt', 'r') as f:
    print(f.read())

Summary

This dataset (ml-latest-small) describes 5-star rating and free-text tagging activity from [MovieLens](http://movielens.org), a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies. These data were created by 610 users between March 29, 1996 and September 24, 2018. This dataset was generated on September 26, 2018.

Users were selected at random for inclusion. All selected users had rated at least 20 movies. No demographic information is included. Each user is represented by an id, and no other information is provided.

The data are contained in the files `links.csv`, `movies.csv`, `ratings.csv` and `tags.csv`. More details about the contents and use of all these files follows.

This is a *development* dataset. As such, it may change over time and is not an appropriate dataset for shared research results. See available *benchmark* datasets if that is your intent.

This and other GroupLens data sets are publicly available for down

In [78]:
ratings = pd.read_csv("/content/ml-latest-small/ratings.csv", sep=",")
print(ratings.shape)
ratings.head()

(100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [79]:
links = pd.read_csv("/content/ml-latest-small/links.csv", sep=",")
print(links.shape)
links.head()

(9742, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [80]:
movies = pd.read_csv("/content/ml-latest-small/movies.csv", sep=",")
print(movies.shape)
movies.head()

(9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [81]:
tags = pd.read_csv("/content/ml-latest-small/tags.csv", sep=",")
print(tags.shape)
tags.head()

(3683, 4)


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


## Create User Item Rating Map
This map will be useful later.

In [82]:
# much faster than df.iloc loop
rating_map = {f"{u}_{m}": r for u, m, r in zip(ratings['userId'], ratings['movieId'], ratings['rating'])}

In [83]:
iterator = iter(rating_map.items())
for i in range(5):
    print(next(iterator))

('1_1', 4.0)
('1_3', 4.0)
('1_6', 4.0)
('1_47', 5.0)
('1_50', 5.0)


## Create movie genre map
This map will also be useful later

In [84]:
movie_genres = {m: set(g.split('|')) for m, g in zip(movies['movieId'], movies['genres'])}

In [85]:
iterator = iter(movie_genres.items())
for i in range(5):
    print(next(iterator))

(1, {'Comedy', 'Animation', 'Fantasy', 'Adventure', 'Children'})
(2, {'Fantasy', 'Adventure', 'Children'})
(3, {'Comedy', 'Romance'})
(4, {'Comedy', 'Romance', 'Drama'})
(5, {'Comedy'})


In [86]:
!pip install -q sentence-transformers

## Generate Transformer Embeddings
We concatenate the title, genres, and tags for each movie into a single string and encode them into dense vectors using a pre-trained sentence transformer model.

In [87]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Group tags by movieId and join them into a single string
grouped_tags = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(map(str, x))).reset_index()

# Merge movies with their tags
movies_enriched = movies.merge(grouped_tags, on='movieId', how='left')
movies_enriched['tag'] = movies_enriched['tag'].fillna('')

# Concatenate title, genres (replacing '|' with spaces), and tags
movies_enriched['combined_features'] = (
    movies_enriched['title'] + " " +
    movies_enriched['genres'].str.replace('|', ' ', regex=False) + " " +
    movies_enriched['tag']
)

# Initialize a lightweight, fast Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
print("Encoding movie features... (this might take a moment)")
embeddings = model.encode(movies_enriched['combined_features'].tolist(), show_progress_bar=True)

# Store in a dictionary for fast O(1) access
movie_embeddings = {m: emb for m, emb in zip(movies_enriched['movieId'], embeddings)}
print("Embeddings successfully created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding movie features... (this might take a moment)


Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Embeddings successfully created!


In [88]:
movies_enriched.head()

,movieId,title,genres,tag,combined_features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,Jumanji (1995) Adventure Children Fantasy fant...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Grumpier Old Men (1995) Comedy Romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,Waiting to Exhale (1995) Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Father of the Bride Part II (1995) Comedy preg...


In [89]:
movies_enriched.iloc[0]["combined_features"]

'Toy Story (1995) Adventure Animation Children Comedy Fantasy pixar pixar fun'

In [90]:
print(len(embeddings[0]))
embeddings[0]

384


array([-7.05864802e-02,  1.53667014e-02,  3.46915051e-02, -3.23326513e-02,
       -3.33275646e-02,  3.66937392e-03,  6.92035258e-02, -9.53246932e-03,
        6.42089732e-03,  5.52398898e-03,  3.09790596e-02,  1.44268421e-03,
        9.54519864e-03,  7.73921162e-02,  1.16008809e-02,  2.88104173e-03,
        3.72703224e-02,  4.85197417e-02,  8.40884224e-02, -6.34905472e-02,
        6.46939725e-02,  1.82996946e-03,  5.32270856e-02, -3.82752903e-02,
       -8.12973082e-02,  1.57797616e-02, -1.73576195e-02, -4.56608124e-02,
       -6.53539225e-02, -2.45841220e-02,  7.38992169e-02,  8.11134875e-02,
        6.54440466e-03, -6.89608976e-02, -1.76655035e-03,  5.32460446e-03,
        9.20672622e-03, -7.50869215e-02,  1.69543251e-02,  1.38746481e-02,
       -4.32794578e-02,  5.44952600e-05,  2.18995661e-02, -5.90835549e-02,
        4.60836478e-02, -1.80067625e-02, -5.07701598e-02, -7.53527656e-02,
        3.58934216e-02,  8.77005383e-02, -7.39513934e-02, -1.61159430e-02,
        8.57316405e-02, -

### How to Interpret the 384-Dimensional Embedding Vectors?

The output above shows a single vector containing **384 float numbers** for a movie. What do they mean?

1. **Latent Semantic Features:** Each of the 384 dimensions represents a "latent" (hidden) feature of the text that the neural network has learned during its training. Unlike explicit genres (where a column might literally mean "Action" or "Comedy"), these dimensions are abstract. One dimension might vaguely correlate with "lighthearted animation," while another captures "dark suspense," but individual dimensions are usually not human-readable.
2. **Holistic Representation:** You cannot look at a single number (e.g., `-0.0705` at index 0) and derive specific meaning from it. The movie's "meaning" is distributed across all 384 numbers. Together, they form a specific coordinate in a 384-dimensional mathematical space.
3. **Spatial Similarity (The Key Concept):** The most important property of these vectors is their relative positioning. The Sentence Transformer is trained so that **semantically similar texts are placed close together in this space**. If two movies have similar descriptions, genres, and tags, their vectors will point in roughly the same direction. We measure this closeness using distance metrics like **Cosine Similarity** to find recommendations.

## Inspecting Nearest Neighbors (Embeddings)
Let's inspect the nearest neighbors of a well-known movie to see if the neighbors make intuitive sense.

In [91]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

target_movieId = 1  # Toy Story (1995)

print("--- Target Movie ---")
display(movies_enriched[movies_enriched['movieId'] == target_movieId][['movieId', 'title', 'genres', 'tag']])

print("\n--- Top 10 Nearest Neighbors (On-the-fly Embeddings) ---")

# Get target embedding
target_emb = movie_embeddings[target_movieId].reshape(1, -1)

# Get all embeddings
movie_ids = list(movie_embeddings.keys())
all_embs = np.array(list(movie_embeddings.values()))

# Compute cosine similarity between the target movie and all other movies
sim_scores = cosine_similarity(target_emb, all_embs)[0]

# Get top indices sorted by similarity (excluding the movie itself)
top_indices = np.argsort(sim_scores)[::-1]

neighbor_ids = []
neighbor_scores = []
for idx in top_indices:
    m_id = movie_ids[idx]
    if m_id != target_movieId and len(neighbor_ids) < 10:
        neighbor_ids.append(m_id)
        neighbor_scores.append(sim_scores[idx])

# Fetch movie details for the neighbors
neighbor_df = movies_enriched[movies_enriched['movieId'].isin(neighbor_ids)].copy()

# Map the similarity scores and sort
neighbor_df['sim_score'] = neighbor_df['movieId'].map(dict(zip(neighbor_ids, neighbor_scores)))
neighbor_df = neighbor_df.sort_values('sim_score', ascending=False)

display(neighbor_df[['movieId', 'title', 'genres', 'tag', 'sim_score']])

--- Target Movie ---


,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun



--- Top 10 Nearest Neighbors (On-the-fly Embeddings) ---


,movieId,title,genres,tag,sim_score
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,animation Disney funny original Pixar sequel T...,0.798101
1757,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,Pixar,0.748133
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,,0.743558
8357,108932,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fan...,cheeky clever colorful feel-good fun imaginati...,0.728797
205,239,"Goofy Movie, A (1995)",Animation|Children|Comedy|Romance,,0.714060
9120,145935,"Peanuts Movie, The (2015)",Adventure|Animation|Children|Comedy,,0.691869
8219,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,,0.678327
6194,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,,0.666672
1674,2253,Toys (1992),Comedy|Fantasy,,0.661688
2572,3440,Teenage Mutant Ninja Turtles III (1993),Action|Adventure|Children|Comedy|Fantasy,,0.660349


### Do these make sense?
By looking at the results for *Toy Story (1995)*, we can judge the quality of the embeddings:
1. **Semantic Understanding:** The embeddings often successfully pull in other Pixar films (like *Toy Story 2*, *Bug's Life*, *Monsters, Inc.*) because of the presence of tags like 'pixar' and 'animation', as well as the inherent semantic similarity in their textual features.
2. **Beyond Strict Genres:** Unlike pure Jaccard similarity which only looks at exact genre matches (e.g., `Adventure|Animation|Children|Comedy|Fantasy`), the embeddings can find movies that are conceptually similar even if a genre tag is missing or slightly different, because the underlying vector space groups 'family-friendly animated comedies' closely together.

## Rating Prediction

### Algorithm

Predict rating of user $u$ for item $i$
- Calculate the similarity of items that are rated by $u$ with $i$.
- Use these similarities to calculate a weighted average of the ratings.

Below we only use the genres to calculate the content similarity between movies, but in general, one can use many other information such as the director, date, casting and plot summary. In order to do this one needs to find a way to represent this information and a similarity metric to quantify the similarity.

Let us first see an example.

In [92]:
movies_ratings_join = movies.merge(ratings, left_on="movieId", right_on="movieId")

In [93]:
movies_ratings_join.query("userId == 1").head(5)

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
325,3,Grumpier Old Men (1995),Comedy|Romance,1,4.0,964981247
433,6,Heat (1995),Action|Crime|Thriller,1,4.0,964982224
2107,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,1,5.0,964983815
2379,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,1,5.0,964982931


What would be the rating of this user for the following movie?

In [94]:
movies_ratings_join.query("userId == 1 and movieId == 70")

,movieId,title,genres,userId,rating,timestamp
2859,70,From Dusk Till Dawn (1996),Action|Comedy|Horror|Thriller,1,3.0,964982400


Following is the calculation for this example by using Jaccard similarity betwee movie jenres:

Jaccard(Toy Story, From Dusk Till Dawn) = 1/8 <br>
Jaccard(Grumpier Old Men, From Dusk Till Dawn) = 1/5 <br>
Jaccard(Heat, From Dusk Till Dawn) = 2/5 <br>
Jaccard(Seven (a.k.a. Se7en), From Dusk Till Dawn) = 1/5 <br>
Jaccard(Usual Suspects, From Dusk Till Dawn) = 1/6<br>


In [95]:
prediction = (1/8*4+1/5*4+2/5*4+1/5*5+1/6*5)/(1/8+1/5+2/5+1/5+1/6)
prediction

4.335877862595419

### Jaccard Similarity

Given two sets $A$ and $B$,

$Jaccard(A, B) = \frac{|A \cap B|}{|A \cup B|}$

For example if $A = \{a, b, c, d\}$ and $B = \{b, d, e ,f, g\}$ then

$Jaccard(A, B) = \frac{2}{7}$

In [96]:
from numpy.linalg import norm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import pandas as pd
import heapq
from tqdm.notebook import tqdm

## Similarity & Prediction Functions
Here we define our core functions to support the `sim_type` parameter. It accepts either `'genre'` (using Jaccard similarity) or `'embedding'` (using Cosine similarity).

In [97]:
def item_sim(i, j, sim_type='genre'):
    if sim_type == 'genre':
        genres_i = movie_genres.get(i, set())
        genres_j = movie_genres.get(j, set())

        intersection_size = len(genres_i.intersection(genres_j))
        union_size = len(genres_i.union(genres_j))
        return intersection_size / union_size if union_size > 0 else 0.0

    elif sim_type == 'embedding':
        emb_i = movie_embeddings.get(i)
        emb_j = movie_embeddings.get(j)
        if emb_i is None or emb_j is None:
            return 0.0
        # Cosine similarity between two vectors
        return np.dot(emb_i, emb_j) / (norm(emb_i) * norm(emb_j) + 1e-9)
    else:
        raise ValueError("sim_type must be either 'genre' or 'embedding'")

In [98]:
def content_based_rating_prediction(u, i, sim_type='genre'):
    r = 0
    sum_sim = 0
    movies_rated = ratings[ratings["userId"]==u].movieId
    for j in movies_rated:
        sim = item_sim(i, j, sim_type=sim_type)
        key = f"{u}_{j}"
        if key in rating_map:
            r += sim * rating_map[key]
            sum_sim += sim

    return r / sum_sim if sum_sim > 0 else 0

In [99]:
content_based_rating_prediction(20,101, sim_type='genre')

3.56100402175646

In [100]:
content_based_rating_prediction(20,101, sim_type='embedding')

np.float32(3.4986007)

## Evaluation of Rating Prediction

How can we measure the performance of a recommender algorithm? This is similar to the evaluation used in machine learning.

- Make a train/test split
- Build the model on the training set
- Make predictions for the ratings in the test set
- Find the mean absolute error (MAE)

For more metrics other then MAE look at the "Metrics for Regression" section of [this notebook](http://localhost:8888/notebooks/PycharmProjects/data_science/evaluation.ipynb)


In [101]:
X_train, X_test = train_test_split(ratings, test_size=1000)
train_size = X_train.shape[0]
test_size = X_test.shape[0]
print("Test size:", test_size)
error_e = 0
error_g = 0
for k in tqdm(range(test_size)):
    u = X_test.iloc[k,0]
    i = X_test.iloc[k,1]
    r = X_test.iloc[k,2]
    error_g += np.abs(r - content_based_rating_prediction(u,i,sim_type='genre'))
    error_e += np.abs(r - content_based_rating_prediction(u,i,sim_type='embedding'))
print("Error - genre:", error_g/test_size)
print("Error - embedding:", error_e/test_size)


Test size: 1000


  0%|          | 0/1000 [00:00<?, ?it/s]

Error - genre: 0.678839951095094
Error - embedding: 0.7103133075237275


## Why Embeddings Don't Always Improve Rating Prediction

It's a common observation that richer semantic representations (embeddings) do not always dramatically lower the Mean Absolute Error (MAE) in **rating prediction** compared to simple genre matching. Here is why:

### 1. Rating Prediction vs. Ranking (Top-N)
Predicting an exact rating (e.g., 3.5 vs 4.0) is a regression task, heavily influenced by user bias (some users rate everything a 4) and item bias (popular movies have higher average scores). Both Genre Jaccard and Embedding Cosine similarities act simply as *weights* for a weighted average. Whether a past rating is weighted by `0.6` (Genre) or `0.75` (Embedding), the final predicted rating often gravitates toward the user's historical average anyway, making the MAE look very similar.

However, if you look at the **Top-N Recommendation (Hit Ratio)** evaluation, **embeddings actually performed twice as well as genres** (Hit Ratio: ~0.26 vs ~0.12)! Embeddings are much better at *ranking* items (deciding what a user will actually watch) than predicting the exact scalar value of the rating.

### 2. Feature Sparsity
The embeddings in this notebook were created by concatenating `Title + Genre + Tags`. Because the `Tags` dataset in MovieLens is very sparse, for a vast majority of movies, the embedding is primarily just encoding the Title and the Genre. Since titles don't always inherently describe the movie's plot to a pre-trained sentence transformer, the strongest signal in the embedding often ends up just being the genre anyway.



## Top-N recommendation Algorithm - Predict and Sort
The task in top-$N$ recommendation is to recommend $N$ items to a user.


Recommend $N$ movies to user $u$
- Predict the ratings of all items which are not watched by $u$
- Sort the predicted ratings
- Recommend the movies with the highest predicted ratings

In [102]:
def top_N_pred_sort(N, u, sim_type='genre'):
    preds = pd.Series([], dtype='float')
    all_items = set(ratings['movieId'].unique())
    rated_by_user = set(ratings[ratings["userId"]==u].movieId.unique())
    not_rated_by_user = all_items - rated_by_user

    for m in tqdm(not_rated_by_user, desc=f"Predicting for user {u} ({sim_type})"):
        preds[m] = content_based_rating_prediction(u, m, sim_type=sim_type)
    return preds.sort_values(ascending=False)[:N]



In [103]:
top_N_pred_sort(10, 1)

Predicting for user 1 (genre):   0%|          | 0/9492 [00:00<?, ?it/s]

,0
7335,5.000000
4426,5.000000
2066,5.000000
69469,4.718136
170597,4.718136
153236,4.718136
163386,4.718136
172909,4.718136
117545,4.718136
81018,4.718136


## Efficiency Issues

There are important inefficiencies in this algorithm:

- The algorithm predicts the rating of all items which are not rated by the user. In the case of millions of items this algorithm is practically infeasible. Numerous techniques have been developed to remedy this problem. Can you suggest a solution?
- In rating prediction, similarity between target item and items rated by the user are calculated. To make a recommendation to another user similarity calculations will be done again. For making recommendations to users in general many similarity calculations will be repeated. A general solution to this problem is to precalculate the similarities between items. Moreover, you don't need to store all similarities, only storing $k$ most similar items to every item will be enough. Size of $k$ can be determined according to the needs.


## Top-N recommendation Algorithm - kNN Map
The task in top-$N$ recommendation is to recommend $N$ items to a user.

- Build a knn-map (a map which stores the $k$ nearest neighbors of each item)

Recommend $N$ movies to user $u$
- Get the neigbors of movies which are watched by $u$ and put them into a list $C$.
- Choose $N$ movies from $C$. There can be different methods here. Most repeated movies in C can be chosen, movies with the highest total similarity (or maximum similarity) can be chosen.
- Recommend the $N$ movies that are chosen.

## Building a knn map
This table will hold the most similar $k$ items for each item. In order to build this table we need to calculate all pairwise similarities which takes $O(n^2)$ time. There is no escape from this $O(n^2)$ time unless you use an approximation algorithm such as LSH (Locality Sensitive Hashing) for nearest neighbor search.

We will use a heap based priority queue for storing the nearest neighbors. You can look at this [animation](https://www.cs.usfca.edu/~galles/visualization/Heap.html).

In [104]:
pq =[(10,"a"),(8, "b"), (5, "c")]
heapq.heapify(pq)
heapq.nsmallest(2,pq)

[(5, 'c'), (8, 'b')]

In [105]:
def build_knn_map(movies, K=10, sim_type='genre'):
    knn_map = {}
    movie_ids = movies['movieId'].unique()

    if sim_type == 'embedding':
        print("Optimizing embedding KNN map with matrix multiplication...")
        emb_matrix = np.array([movie_embeddings.get(m, np.zeros(384)) for m in movie_ids])
        sim_matrix = cosine_similarity(emb_matrix)

    elif sim_type == 'genre':
        print("Optimizing genre KNN map with vectorized Jaccard similarity...")
        mlb = MultiLabelBinarizer()
        genre_lists = [list(movie_genres.get(m, [])) for m in movie_ids]
        genre_matrix = mlb.fit_transform(genre_lists)

        # Intersection = dot product of binary matrices
        intersection = np.dot(genre_matrix, genre_matrix.T)

        # Union = sum(A) + sum(B) - Intersection
        genre_sums = genre_matrix.sum(axis=1)
        union = genre_sums[:, None] + genre_sums[None, :] - intersection

        # Jaccard = Intersection / Union (handling division by zero)
        sim_matrix = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)
    else:
        raise ValueError("sim_type must be 'genre' or 'embedding'")

    # Process the calculated similarity matrix
    for i in tqdm(range(len(movie_ids)), desc=f"Building KNN Map ({sim_type})"):
        m_i = movie_ids[i]
        sims = sim_matrix[i]

        top_indices = np.argpartition(sims, -(K+1))[-(K+1):]
        top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]

        pq = []
        for j in top_indices:
            if i != j and len(pq) < K:
                pq.append((sims[j], movie_ids[j]))

        knn_map[m_i] = pq

    return knn_map

In [106]:
knn_map = build_knn_map(movies,K=30)

Optimizing genre KNN map with vectorized Jaccard similarity...


Building KNN Map (genre):   0%|          | 0/9742 [00:00<?, ?it/s]

In [107]:
knn_map[3]

[(np.float64(1.0), np.int64(252)),
 (np.float64(1.0), np.int64(4782)),
 (np.float64(1.0), np.int64(50189)),
 (np.float64(1.0), np.int64(50806)),
 (np.float64(1.0), np.int64(4639)),
 (np.float64(1.0), np.int64(4672)),
 (np.float64(1.0), np.int64(51705)),
 (np.float64(1.0), np.int64(4743)),
 (np.float64(1.0), np.int64(122)),
 (np.float64(1.0), np.int64(53974)),
 (np.float64(1.0), np.int64(186)),
 (np.float64(1.0), np.int64(52867)),
 (np.float64(1.0), np.int64(188751)),
 (np.float64(1.0), np.int64(39)),
 (np.float64(1.0), np.int64(51082)),
 (np.float64(1.0), np.int64(50954)),
 (np.float64(1.0), np.int64(4740)),
 (np.float64(1.0), np.int64(8336)),
 (np.float64(1.0), np.int64(139915)),
 (np.float64(1.0), np.int64(140301)),
 (np.float64(1.0), np.int64(7)),
 (np.float64(1.0), np.int64(2339)),
 (np.float64(1.0), np.int64(133712)),
 (np.float64(1.0), np.int64(132488)),
 (np.float64(1.0), np.int64(4471)),
 (np.float64(1.0), np.int64(5135)),
 (np.float64(1.0), np.int64(4877)),
 (np.float64(1.0), 

## Top-N recommendation using knn map

In [108]:
def top_N_knn_map(ratings, N, u):
    C = []
    # find the movies rated by u
    movies_rated = ratings.query("userId == @u").movieId
    for m in movies_rated:
        C = C + knn_map[m]
    return add_sims_and_sort(C)[:N]

In [109]:
def add_sims_and_sort(l):
    li = []
    it = itertools.groupby(l, operator.itemgetter(1))
    for key, subiter in it:
        li.append((key, sum(item[0] for item in subiter)))
    li = sorted(li, key=itemgetter(1), reverse=True)
    return li


In [110]:
top_N_knn_map(ratings, 10, 1)

[(np.int64(3754), np.float64(1.0)),
 (np.int64(4016), np.float64(1.0)),
 (np.int64(53121), np.float64(1.0)),
 (np.int64(2294), np.float64(1.0)),
 (np.int64(136016), np.float64(1.0)),
 (np.int64(91355), np.float64(1.0)),
 (np.int64(3114), np.float64(1.0)),
 (np.int64(166461), np.float64(1.0)),
 (np.int64(103755), np.float64(1.0)),
 (np.int64(4886), np.float64(1.0))]

## Evaluation of top-N recommendation

Evaluation of rating prediction is rather easy: find the mean absolute error between rating predictions and true ratings. How can we evaluate the accuracy of a top-N recommendation? There are several techniques which we will look at in more detail later. Below is one common way to evaluate top-N recommendation:

- Randomly sub-sample some portion of positive preferences in order to create a test set $T$. Positive preferences might be 5-star ratings, movies watched more than a certain threshold, or items purchased.
- Put the rest of the preferences into the training set and build model.

- For each preference $(u,i)$ in the test set:
    - Make a top-N recommendation tu user $u$.
    - If the test item i occurs among the top-N items, then we have a hit, otherwise we have a miss.

Hit ratio is then defined as:

$$
Hit Ratio: \frac{\#hits}{|T|}
$$



In [111]:
N = 100
X_train, X_test = train_test_split(ratings, test_size=1000)
X_test = X_test.query("rating > 4")
train_size = X_train.shape[0]
test_size = X_test.shape[0]
print("Test size:", test_size)
hit_count = 0
for k in tqdm(range(test_size)):
    u = X_test.iloc[k,0]
    i = X_test.iloc[k,1]
    r = X_test.iloc[k,2]
    top_N = top_N_knn_map(X_train, N, u)
    hit_list = [item for item in top_N if item[0] == i]
    if len(hit_list) > 0:
        hit_count +=1
print("Hit Ratio", hit_count/test_size)

Test size: 215


  0%|          | 0/215 [00:00<?, ?it/s]

Hit Ratio 0.013953488372093023


## Evaluate and Compare: Genre vs Embedding

In [112]:
# 1. Build KNN maps for both approaches
print("Building KNN map for genres...")
knn_map_genre = build_knn_map(movies, K=30, sim_type='genre')

print("Building KNN map for embeddings...")
knn_map_embedding = build_knn_map(movies, K=30, sim_type='embedding')

Building KNN map for genres...
Optimizing genre KNN map with vectorized Jaccard similarity...


Building KNN Map (genre):   0%|          | 0/9742 [00:00<?, ?it/s]

Building KNN map for embeddings...
Optimizing embedding KNN map with matrix multiplication...


Building KNN Map (embedding):   0%|          | 0/9742 [00:00<?, ?it/s]

In [113]:
# 2. Update top_N_knn_map to accept a precomputed list of rated movies
def top_N_knn_map_fast(rated_movies, N, specific_knn_map):
    C = []
    for m in rated_movies:
        if m in specific_knn_map:
            C = C + specific_knn_map[m]
    return add_sims_and_sort(C)[:N]

In [125]:
import math
import pandas as pd

# 3. Run evaluation
N = 20

# Identify positive preferences (ratings > 4) and sort by timestamp to respect causality
positive_ratings = ratings.query("rating > 4").sort_values('timestamp')

# Time-based Leave-One-Out: get the last positive rating for each user
# We use all users who have at least one positive rating
X_test = positive_ratings.groupby('userId').tail(1)

# The training set should contain EVERYTHING else in the dataset (no ratings wasted)
X_train = ratings.drop(X_test.index)


test_size = X_test.shape[0]
print("Test size (rating > 4, time-based leave-one-out):", test_size)

# OPTIMIZATION: Pre-compute user histories to avoid slow pandas queries inside the loop
# Only put movies that the user rated X or higher in the user histories (what should be the value of X?).
positive_X_train = X_train.query("rating >= 4")
user_histories = positive_X_train.groupby('userId')['movieId'].apply(list).to_dict()

hit_count_genre = 0
hit_count_embedding = 0
ndcg_genre = 0.0
ndcg_embedding = 0.0

for k in tqdm(range(test_size), desc="Evaluating Hit Ratio & NDCG"):
    u = X_test.iloc[k,0]
    i = X_test.iloc[k,1]

    # Get the user's precomputed positive watch history
    rated_movies = user_histories.get(u, [])

    # Evaluate Genre
    top_N_genre = top_N_knn_map_fast(rated_movies, N, knn_map_genre)
    rank_genre = next((idx for idx, item in enumerate(top_N_genre) if item[0] == i), -1)
    if rank_genre != -1:
        hit_count_genre += 1
        ndcg_genre += 1.0 / math.log2(rank_genre + 2) # rank is 0-indexed, so we use rank+2 for log base 2

    # Evaluate Embedding
    top_N_embedding = top_N_knn_map_fast(rated_movies, N, knn_map_embedding)
    rank_embedding = next((idx for idx, item in enumerate(top_N_embedding) if item[0] == i), -1)
    if rank_embedding != -1:
        hit_count_embedding += 1
        ndcg_embedding += 1.0 / math.log2(rank_embedding + 2)

print(f"\n--- Evaluation Complete ---")
results_df = pd.DataFrame({
    "Method": ["Genre (Jaccard)", "Embedding (Cosine)"],
    f"Hit Ratio@{N}": [hit_count_genre/test_size, hit_count_embedding/test_size],
    f"NDCG@{N}": [ndcg_genre/test_size, ndcg_embedding/test_size],
    "Total Hits": [hit_count_genre, hit_count_embedding]
})

display(results_df)


Test size (rating > 4, time-based leave-one-out): 598


Evaluating Hit Ratio & NDCG:   0%|          | 0/598 [00:00<?, ?it/s]


--- Evaluation Complete ---


,Method,Hit Ratio@20,NDCG@20,Total Hits
0,Genre (Jaccard),0.006689,0.001853,4
1,Embedding (Cosine),0.040134,0.015902,24


## Why does Item-Based Collaborative Filtering outperform Content-Based (Embeddings)?

You might notice that the Hit Ratio using Embeddings is better than Genres, but it still falls short of Item-Based Collaborative Filtering. Here is why this happens:

1. **Behavioral vs. Textual Signals (The Wisdom of the Crowd):**
Collaborative Filtering (CF) uses actual user behavior (ratings). If thousands of users who liked *Toy Story* also highly rated *Star Wars*, CF learns this connection instantly. Content-Based filtering only looks at metadata (Genres, Tags). Human behavior naturally captures complex, unspoken preferences that simple text features miss.

2. **The Concept of "Quality":**
Content-based filtering has no sense of whether a movie is actually *good*. It will confidently recommend a terrible, low-budget sci-fi movie alongside a sci-fi masterpiece simply because their genre tags and plot keywords match. CF, on the other hand, naturally filters for quality because it relies on what similar users actually enjoyed.

3. **Feature Sparsity:**
The embeddings in this notebook rely on a very limited set of text: Title, Genres, and sparse User Tags. Without rich plot summaries, cast lists, directors, and critical reviews, the language model's understanding of the movie is quite shallow compared to the deep interaction history encoded in the user-item rating matrix.

4. **Serendipity and Hidden Tastes:**
CF can find non-obvious connections. For example, it might figure out that fans of a specific dark comedy also tend to like a specific gritty crime thriller. Text embeddings struggle to make this leap unless the semantic text of both movies explicitly bridges that gap.

**Conclusion:**
Embeddings and Content-Based methods are fantastic for solving the **"Cold Start" problem** (recommending brand new movies that have zero ratings yet), but when a rich history of user rating data is available, Collaborative Filtering almost always wins because *human behavior is the strongest predictor of future human behavior*.

## Is Content Still Useful When We Have Enough Ratings?

If Item-Based Collaborative Filtering (CF) is so powerful when we have enough data, do we just throw away content-based methods once we pass the "cold-start" phase? **Absolutely not.** Content-based features remain extremely valuable for several reasons:

1. **Overcoming Popularity Bias (The Long Tail):**
CF algorithms naturally suffer from popularity bias. Blockbuster movies like *The Matrix* or *Forrest Gump* have so many ratings that they artificially overlap with almost every user's history. CF often ends up just recommending blockbusters. Content-based embeddings don't care how many ratings a movie has; they can surface a "hidden gem" (a niche movie with only 5 ratings) that perfectly matches the semantics of what you love.

2. **Explainability:**
Content-based methods provide natural explanations. A system can say, *"We recommend this because it shares the 'cyberpunk' and 'dystopia' tags with The Matrix."* CF explanations are often opaque (*"We recommend this because 500 users who vaguely share your taste liked it"*), which doesn't build as much user trust.

3. **Catering to Specific Moods:**
Sometimes a user is in the mood for a very specific type of movie (e.g., a "dark comedy heist movie"). CF might look at the user's recent history and recommend a drama just because it's highly rated by their peers. Content vectors allow users to search and filter dynamically based on textual themes.

4. **Hybrid Recommender Systems:**
In the real world (Netflix, Spotify, Amazon), companies never use just one method. They build **Hybrid Systems**. A common architecture is to use CF to generate a pool of 1,000 high-quality candidate movies that the user's peers enjoyed, and then use a Content-Based deep learning model (like embeddings) to heavily personalize and re-rank the final Top 10 based on the exact features, genres, and directors the user engages with most.

## Exercises

**Exercise 1: Basic Rating Prediction**
Using the `content_based_rating_prediction` function defined earlier, predict the rating that User `2` would give to the movie *Toy Story (1995)* (which has `movieId = 1`). Use the `embedding` similarity type.

**Exercise 2: k-Nearest Neighbors (kNN) Rating Prediction**
The current `content_based_rating_prediction` function uses *all* movies rated by the user to calculate the weighted average. Modify this logic to create a new function `content_based_rating_prediction_knn(u, i, sim_type='embedding', k=5)` that only uses the top `k` most similar movies the user has rated to compute the prediction.

**Exercise 3: Finding Similar Movies**
Using the `movie_embeddings` dictionary and `cosine_similarity`, find and display the top 5 nearest neighbors for the movie *The Matrix (1999)* (which has `movieId = 2571`). Do not include the movie itself in the results.

## Conceptual Exercises (No Coding Required)

**Exercise 4: The Cold Start Problem**
If a brand-new movie is added to the MovieLens dataset today and zero users have rated it yet, which recommendation approach discussed in this notebook would be able to recommend it to users: Collaborative Filtering or Content-Based (Embeddings)? Why?

**Exercise 5: Understanding Metrics**
Why might the Embedding-based approach score a much better **Hit Ratio** than the Genre-based approach, even if both methods yield very similar **Mean Absolute Error (MAE)** scores during rating prediction?

**Exercise 6: Feature Limitations**
Our embedding model currently concatenates `Title`, `Genres`, and `Tags`. Name two additional pieces of movie metadata that, if added to the text string, would likely significantly improve the quality and semantic meaning of the movie embeddings.

**Exercise 7: Rank-Aware Evaluation (NDCG)**
Suppose Method A and Method B both achieve the exact same **Hit Ratio@100** on our test set. However, Method A has a significantly higher **NDCG@100** than Method B. What does this tell us about how Method A ranks its recommendations compared to Method B?

**Exercise 8: Calculating NDCG by Hand**
Suppose our recommendation model outputs a Top-10 list for a user. The user's actual watched movie (the "hit") appears at **rank 3** (1-indexed, meaning it is the 3rd movie in the list).
1. Using the NDCG formula applied in this notebook's evaluation loop (where $\text{NDCG} = 1 / \log_2(\text{rank} + 1)$ for a 1-indexed rank), calculate the NDCG score for this hit.
2. What would the NDCG score be if the movie was placed perfectly at **rank 1**?
3. What would the NDCG score be if the movie was placed at **rank 7**?

## Solutions

### Solution 1: Basic Rating Prediction

In [115]:
user_id = 2
movie_id = 1 # Toy Story

predicted_rating = content_based_rating_prediction(user_id, movie_id, sim_type='embedding')
print(f"Predicted rating for User {user_id} on Movie {movie_id} (Toy Story) is: {predicted_rating:.2f}")

Predicted rating for User 2 on Movie 1 (Toy Story) is: 3.92


### Solution 2: k-Nearest Neighbors (kNN) Rating Prediction

In [116]:
def content_based_rating_prediction_knn(u, i, sim_type='embedding', k=5):
    movies_rated = ratings[ratings["userId"]==u].movieId
    sims_and_ratings = []

    for j in movies_rated:
        sim = item_sim(i, j, sim_type=sim_type)
        key = f"{u}_{j}"
        if key in rating_map:
            sims_and_ratings.append((sim, rating_map[key]))

    # Sort by similarity in descending order and keep top k
    sims_and_ratings.sort(key=lambda x: x[0], reverse=True)
    top_k = sims_and_ratings[:k]

    r = 0
    sum_sim = 0
    for sim, rating in top_k:
        r += sim * rating
        sum_sim += sim

    return r / sum_sim if sum_sim > 0 else 0

# Test the new function
pred_knn = content_based_rating_prediction_knn(user_id, movie_id, sim_type='embedding', k=5)
print(f"Predicted rating (kNN, k=5) for User {user_id} on Movie {movie_id} is: {pred_knn:.2f}")

Predicted rating (kNN, k=5) for User 2 on Movie 1 is: 3.86


### Solution 3: Finding Similar Movies (The Matrix)

In [117]:
target_movieId = 2571  # The Matrix (1999)
target_emb = movie_embeddings[target_movieId].reshape(1, -1)

# Compute cosine similarities
sim_scores = cosine_similarity(target_emb, all_embs)[0]

# Get indices sorted by highest similarity first
top_indices = np.argsort(sim_scores)[::-1]

neighbor_ids = []
neighbor_scores = []
for idx in top_indices:
    m_id = movie_ids[idx]
    if m_id != target_movieId and len(neighbor_ids) < 5:
        neighbor_ids.append(m_id)
        neighbor_scores.append(sim_scores[idx])

# Filter the dataframe to show these top 5 movies
neighbor_df = movies_enriched[movies_enriched['movieId'].isin(neighbor_ids)].copy()
neighbor_df['sim_score'] = neighbor_df['movieId'].map(dict(zip(neighbor_ids, neighbor_scores)))
neighbor_df = neighbor_df.sort_values('sim_score', ascending=False)

print("Top 5 Similar Movies to 'The Matrix (1999)':")
display(neighbor_df[['movieId', 'title', 'genres', 'sim_score']])

Top 5 Similar Movies to 'The Matrix (1999)':


,movieId,title,genres,sim_score
4639,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,0.658598
4351,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,0.610583
2898,3879,"Art of War, The (2000)",Action|Thriller,0.571435
9722,189547,Iron Soldier (2010),Action|Sci-Fi,0.559296
6211,45503,Peaceful Warrior (2006),Drama,0.557715


### Solution 4: The Cold Start Problem
**Content-Based (Embeddings)** would be able to recommend it. This is known as the "Cold Start" problem. Collaborative filtering relies strictly on historical user ratings; without ratings, it cannot find similar items. Content-based methods only need the item's metadata (title, genres, tags) to compute embeddings and find similarities, so they work immediately on new items.

### Solution 5: Understanding Metrics
The two metrics measure different things, so they can disagree.

**MAE** only checks rating accuracy on items the user already rated. Ratings cluster in a narrow range (mostly 3-5), so almost any method that predicts something near the user's or item's average gets a low MAE. A genre-based predictor and an embedding-based one can both land near that average and look alike, even though one understands the items much better. MAE also ignores whether the ordering of items is right, since small score differences barely change the error.

**Hit Ratio** measures ranking across the whole catalog. To get a hit, the relevant item must score above thousands of other items, including ones the user never rated. That depends on fine-grained discrimination, not on being close to the right rating. Genre is coarse: many movies share the same genres, so they get tied or nearly tied scores, and the true item gets buried among them. Embeddings capture finer content (plot, tone, themes, cast), so they separate similar-genre items and push the relevant one higher.

A related factor is the evaluation sample. MAE is computed only on observed ratings, which are biased toward items users chose to watch, while Hit Ratio forces the model to compete against all candidates. A method can predict "a 4 for this drama" correctly and still be unable to say which of 500 dramas the user would pick next.

### Solution 6: Feature Limitations
Adding a **Plot Summary (Synopsis)**, **Cast / Lead Actors**, or **Director** would greatly improve the embeddings. A detailed plot summary would give the Sentence Transformer deep semantic context about the story, character arcs, and setting, rather than relying solely on high-level genre tags.

### Solution 7: Rank-Aware Evaluation (NDCG)
This tells us that Method A is placing the relevant (correct) movies much **higher up** in the Top-100 list than Method B.

**Hit Ratio** is a binary metric: it only checks *if* the target movie is anywhere in the top 100. A hit at rank 1 counts exactly the same as a hit at rank 100.

**NDCG** (Normalized Discounted Cumulative Gain) is a *rank-aware* metric. It applies a logarithmic discount to hits based on their position. A hit at rank 1 contributes heavily to the NDCG score, while a hit at rank 99 contributes very little. Therefore, if Method A has a higher NDCG despite an identical Hit Ratio, it means Method A is practically much more useful to a user because they don't have to scroll as far down the list to find a movie they want to watch.

### Solution 8: Calculating NDCG by Hand
The formula applies a logarithmic discount based on the item's rank: $\text{NDCG} = 1 / \log_2(\text{rank} + 1)$.

1. **Rank 3:** $\text{NDCG} = 1 / \log_2(3 + 1) = 1 / \log_2(4) = 1 / 2 = \mathbf{0.5}$
2. **Rank 1:** $\text{NDCG} = 1 / \log_2(1 + 1) = 1 / \log_2(2) = 1 / 1 = \mathbf{1.0}$
3. **Rank 7:** $\text{NDCG} = 1 / \log_2(7 + 1) = 1 / \log_2(8) = 1 / 3 = \mathbf{0.333}$

Notice how a perfect recommendation gets a score of 1.0, and the score drops off rapidly (0.5 for rank 3, 0.333 for rank 7), heavily rewarding models that place relevant items near the very top of the list.